# PathSnap ML Training & Hyperparameter Tuning Pipeline
### **Member B: Machine Learning Models & Optimization**
**Project:** PathSnap – GPS Map Matching & Route Reconstruction  
**Environment:** Kaggle / Local Reproducible ML Environment  

---

### **Table of Contents**
1. **Introduction & Problem Formulation**
2. **Environment Setup & Dependencies**
3. **Data Loading & Outlier Filtering** (Trips 33 & 54 Removal)
4. **Exploratory Data Analysis on Features**
5. **Candidate Ranking & Top-1 Accuracy Implementation**
6. **Cross-Validation Strategy: Leakage-Free GroupKFold**
7. **Training & Benchmarking 5 Classifiers:**
   - Model 1: Logistic Regression
   - Model 2: K-Nearest Neighbors (KNN - 200k Subsample)
   - Model 3: Random Forest Classifier
   - Model 4: LightGBM Classifier
   - Model 5: Multi-Layer Perceptron (MLP Neural Network)
8. **Comparative Analysis & Visualizations**
9. **Bayesian Hyperparameter Tuning with Optuna**
10. **Final Production Model Training & Artifact Export**
11. **Team Integration Guide (Members C & D)**


## 1. Environment Setup & Library Imports
We configure reproducibility seeds (`random_state=42`), multi-threading parameters, and import standard scientific libraries.


In [ ]:
import os
import sys
import time
import json
import logging
from pathlib import Path
import warnings

# Suppress minor warnings for clean presentation
warnings.filterwarnings('ignore')
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-learn & LightGBM
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, log_loss, accuracy_score
import lightgbm as lgb
import optuna

print("All dependencies successfully imported!")
print(f"NumPy: {np.__version__} | Pandas: {pd.__version__} | LightGBM: {lgb.__version__} | Optuna: {optuna.__version__}")


## 2. Feature Schema & Column Definitions
Based on the candidate generation module (`features/build_features.py`):
- **Candidate Generator:** For each GPS point, road segments within a **50m radius** (up to 8 candidates) are extracted.
- **Identifiers / Grouping:** `trip_id`, `point_idx`, `arc_id`.
- **Target Label:** `label` (binary 1 for true driving road arc, 0 for incorrect candidate).
- **Features ($X$):** 13 spatial and topological features.


In [ ]:
FEATURE_COLS = [
    "dist",                      # Distance from GPS point to candidate road arc (meters)
    "t_frac",                    # Normalized projection point along the candidate segment [0, 1]
    "arc_len",                   # Length of candidate road segment (meters)
    "speed",                     # Vehicle speed derived from GPS track (m/s)
    "dt_prev",                   # Time delta from preceding GPS coordinate (seconds)
    "heading_diff",              # Angular divergence between GPS bearing and road bearing [0, 180 deg]
    "heading_reliable",          # Binary indicator (1 if speed > 1.0 m/s, else 0)
    "n_cand",                    # Total candidate road arcs identified for this GPS point
    "dist_rank",                 # Rank of this candidate by proximity (1 = nearest)
    "dist_minus_min",            # Distance difference compared to nearest candidate (meters)
    "dist_ratio",                # Proximity ratio: dist / (min_dist + 1.0)
    "same_as_prev_nearest",      # 1 if arc is identical to preceding point's nearest candidate
    "connected_to_prev_nearest"  # 1 if arc topologically connects to preceding nearest candidate
]

EXCLUDE_COLS = ["trip_id", "point_idx", "arc_id", "label"]
print(f"Number of engineered features: {len(FEATURE_COLS)}")


## 3. Data Loading & GPS Outlier Removal
### **Why Drop Trips 33 and 54?**
During Data Processing (Member A), exploratory analysis revealed that Trips `00000033` and `00000054` suffered severe GPS sensor drift (~122m and ~140m off the true road network). Because candidate generation uses a **50m search radius**, the true road arc was never captured in the candidate set (candidate miss rate is **67.2%** for trip 33 and **90.7%** for trip 54).

Keeping these trips would train the models on false negatives and distort probability calibration. Filtering them out preserves **78 clean trips** for model training.


In [ ]:
# Automatic path detection: finds parquet files in local repo or ANY Kaggle dataset directory
import glob
from pathlib import Path

# Ensure output directories exist in working directory
Path("results").mkdir(parents=True, exist_ok=True)
Path("artifacts").mkdir(parents=True, exist_ok=True)
Path("tuning").mkdir(parents=True, exist_ok=True)

kaggle_train = glob.glob("/kaggle/input/**/features_train.parquet", recursive=True)
kaggle_test = glob.glob("/kaggle/input/**/features_studenttest.parquet", recursive=True)

if kaggle_train and kaggle_test:
    train_path = Path(kaggle_train[0])
    test_path = Path(kaggle_test[0])
    print(f"[Kaggle Environment] Auto-detected features at: {train_path.parent}")
elif Path("results/features/features_train.parquet").exists():
    train_path = Path("results/features/features_train.parquet")
    test_path = Path("results/features/features_studenttest.parquet")
    print("[Local Environment] Using repository results/features/ folder")
else:
    raise FileNotFoundError("features_train.parquet not found! Please attach dataset.")

df_train_raw = pd.read_parquet(train_path)
df_test = pd.read_parquet(test_path)

print(f"Raw Train Shape: {df_train_raw.shape} ({df_train_raw['trip_id'].nunique()} trips)")
print(f"Student-Test Shape: {df_test.shape} ({df_test['trip_id'].nunique()} trips)")

# Apply outlier trip filter on training set (drop trips 33 and 54 due to >120m GPS drift)
outlier_trips = ['00000033', '00000054']
df_train = df_train_raw[~df_train_raw['trip_id'].isin(outlier_trips)].reset_index(drop=True)

print(f"Clean Train Shape after dropping {outlier_trips}: {df_train.shape} ({df_train['trip_id'].nunique()} trips)")
print(f"Positive Label Proportion (Train): {df_train['label'].mean():.4f} ({df_train['label'].mean()*100:.2f}%)")
print(f"Positive Label Proportion (Student-Test): {df_test['label'].mean():.4f} ({df_test['label'].mean()*100:.2f}%)")


## 4. Candidate Ranking & Top-1 Accuracy
### **Point-Level Top-1 Accuracy vs. Row-Level Accuracy**
- **Row-Level Accuracy** tests whether individual candidates are predicted as 0 or 1. Since ~83% of candidates are negative, predicting all zeros yields 83% accuracy but is useless for map matching.
- **Top-1 Accuracy** matches the real-world downstream goal: For each GPS point $(trip\_id, point\_idx)$, pick the candidate with the highest predicted probability $\hat{P}(label=1)$. The point is correct if and only if that top-ranked candidate is the true road segment ($label=1$).


In [ ]:
def calculate_top1_accuracy(df_meta: pd.DataFrame, probas: np.ndarray) -> float:
    """Calculate Top-1 candidate ranking accuracy grouped by (trip_id, point_idx)."""
    df_eval = pd.DataFrame({
        "trip_id": df_meta["trip_id"].to_numpy(),
        "point_idx": df_meta["point_idx"].to_numpy(),
        "label": df_meta["label"].to_numpy(),
        "prob": np.asarray(probas, dtype=np.float64)
    })
    # Pick candidate with max predicted probability for each GPS point
    idx_max = df_eval.groupby(["trip_id", "point_idx"], sort=False)["prob"].idxmax()
    return float((df_eval.loc[idx_max, "label"] == 1).mean())

def evaluate_predictions(df_meta: pd.DataFrame, y_true: np.ndarray, probas: np.ndarray, threshold: float = 0.5):
    """Compute full suite of classification and ranking metrics."""
    y_true = np.asarray(y_true, dtype=np.int64)
    probas = np.clip(np.asarray(probas, dtype=np.float64), 1e-15, 1.0 - 1e-15)
    y_pred = (probas >= threshold).astype(np.int64)
    
    top1 = calculate_top1_accuracy(df_meta, probas)
    macro_f1 = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    pos_f1 = float(f1_score(y_true, y_pred, average="binary", zero_division=0))
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec = float(recall_score(y_true, y_pred, zero_division=0))
    loss = float(log_loss(y_true, probas))
    row_acc = float(accuracy_score(y_true, y_pred))
    
    return {
        "top1_accuracy": top1,
        "macro_f1": macro_f1,
        "f1": pos_f1,
        "precision": prec,
        "recall": rec,
        "log_loss": loss,
        "row_accuracy": row_acc
    }

# Baseline check: What is Top-1 accuracy if we simply pick the closest candidate arc?
dist_baseline_top1 = calculate_top1_accuracy(df_test[["trip_id", "point_idx", "label"]], -df_test["dist"].to_numpy())
print(f"Geometric Nearest-Distance Baseline Top-1 Accuracy: {dist_baseline_top1:.4f} ({dist_baseline_top1*100:.2f}%)")


## 5. Leak-Free Cross-Validation: GroupKFold
We use `GroupKFold(n_splits=5)` partitioned by `trip_id`. This guarantees that no points from the same trip appear in both training and validation folds.


In [ ]:
X_train = df_train[FEATURE_COLS].copy()
y_train = df_train["label"].to_numpy(dtype=np.int64)
groups_train = df_train["trip_id"].to_numpy()
meta_train = df_train[["trip_id", "point_idx", "arc_id", "label"]].copy()

X_test = df_test[FEATURE_COLS].copy()
y_test = df_test["label"].to_numpy(dtype=np.int64)
meta_test = df_test[["trip_id", "point_idx", "arc_id", "label"]].copy()

print(f"X_train: {X_train.shape} across {len(np.unique(groups_train))} trips")
print(f"X_test:  {X_test.shape} across {len(np.unique(df_test['trip_id']))} trips")


## 6. Training & Benchmarking 5 Classifiers
We train and evaluate:
1. **Logistic Regression** (Standardized Linear Model)
2. **K-Nearest Neighbors (KNN)** (Distance-weighted, evaluated on 200,000 representative training rows)
3. **Random Forest** (Ensemble of Decision Trees)
4. **LightGBM** (Gradient-Boosted Decision Trees)
5. **MLP Classifier** (Multi-Layer Perceptron Neural Network)


In [ ]:
def get_model(name: str):
    name = name.lower()
    if name == "logistic_regression":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced", random_state=42, solver="lbfgs"))
        ])
    elif name == "knn":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=15, weights="distance", algorithm="brute", n_jobs=-1))
        ])
    elif name == "random_forest":
        return RandomForestClassifier(n_estimators=60, max_depth=14, min_samples_leaf=10,
                                      class_weight="balanced_subsample", random_state=42, n_jobs=-1)
    elif name == "lightgbm":
        return lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, max_depth=7,
                                  class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1)
    elif name == "mlp":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu", alpha=1e-4, batch_size=1024,
                                  learning_rate_init=1e-3, max_iter=20, early_stopping=True, n_iter_no_change=3, random_state=42))
        ])
    else:
        raise ValueError(f"Unknown model name: {name}")


In [ ]:
# Run 5-fold GroupKFold benchmark
gkf = GroupKFold(n_splits=5)
models_to_evaluate = ["logistic_regression", "knn", "random_forest", "lightgbm", "mlp"]
benchmark_results = []

for model_name in models_to_evaluate:
    print(f"\n{'='*60}\nTraining Model: {model_name.upper()}\n{'='*60}")
    oof_probas = np.zeros(len(X_train), dtype=np.float64)
    t_train_start = time.time()
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups=groups_train), 1):
        X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
        
        # Subsample KNN training set to 200,000 rows to prevent computational bottleneck
        if model_name == "knn" and len(X_tr) > 200000:
            idx_sub, _ = train_test_split(np.arange(len(X_tr)), train_size=200000, stratify=y_tr, random_state=42 + fold)
            X_tr, y_tr = X_tr.iloc[idx_sub], y_tr[idx_sub]
            
        model = get_model(model_name)
        model.fit(X_tr, y_tr)
        val_probs = model.predict_proba(X_va)[:, 1]
        oof_probas[val_idx] = val_probs
        
    total_train_time = time.time() - t_train_start
    cv_eval = evaluate_predictions(meta_train, y_train, oof_probas)
    
    # Train full model for Student-Test evaluation
    full_model = get_model(model_name)
    X_full, y_full = X_train, y_train
    if model_name == "knn" and len(X_full) > 200000:
        idx_sub, _ = train_test_split(np.arange(len(X_full)), train_size=200000, stratify=y_full, random_state=42)
        X_full, y_full = X_full.iloc[idx_sub], y_full[idx_sub]
    
    full_model.fit(X_full, y_full)
    
    t_test_start = time.time()
    test_probs = full_model.predict_proba(X_test)[:, 1]
    test_infer_time = time.time() - t_test_start
    test_eval = evaluate_predictions(meta_test, y_test, test_probs)
    
    res = {
        "model": model_name,
        "cv_top1_accuracy": cv_eval["top1_accuracy"],
        "cv_macro_f1": cv_eval["macro_f1"],
        "cv_f1": cv_eval["f1"],
        "cv_precision": cv_eval["precision"],
        "cv_recall": cv_eval["recall"],
        "cv_log_loss": cv_eval["log_loss"],
        "test_top1_accuracy": test_eval["top1_accuracy"],
        "test_macro_f1": test_eval["macro_f1"],
        "test_f1": test_eval["f1"],
        "test_precision": test_eval["precision"],
        "test_recall": test_eval["recall"],
        "test_log_loss": test_eval["log_loss"],
        "total_train_time_s": round(total_train_time, 2),
        "test_inference_time_s": round(test_infer_time, 2),
        "latency_ms_per_point": round(1000.0 * test_infer_time / len(X_test), 4)
    }
    benchmark_results.append(res)
    print(f"CV Top-1: {cv_eval['top1_accuracy']:.4f} | Test Top-1: {test_eval['top1_accuracy']:.4f} | Test Macro F1: {test_eval['macro_f1']:.4f}")

df_results = pd.DataFrame(benchmark_results)
df_results.to_csv("results/experiments.csv", index=False)
display(df_results[["model", "cv_top1_accuracy", "cv_macro_f1", "test_top1_accuracy", "test_macro_f1", "cv_log_loss", "total_train_time_s", "test_inference_time_s"]])


## 7. Comparative Performance Visualizations
We generate comparison plots across the 5 models to examine Top-1 ranking accuracy, Macro F1-score, log loss calibration, and latency.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.set_theme(style="whitegrid", palette="muted")

# 1. CV Macro F1
sns.barplot(data=df_results, x="model", y="cv_macro_f1", ax=axes[0, 0], palette="viridis")
axes[0, 0].set_title("5-Fold Grouped CV Macro F1", fontsize=12, weight="bold")
axes[0, 0].set_ylim(0.7, 1.0)

# 2. Student-Test Top-1 Accuracy
sns.barplot(data=df_results, x="model", y="test_top1_accuracy", ax=axes[0, 1], palette="magma")
axes[0, 1].set_title("Student-Test Top-1 Accuracy", fontsize=12, weight="bold")
axes[0, 1].set_ylim(0.7, 1.0)

# 3. Log Loss
sns.barplot(data=df_results, x="model", y="test_log_loss", ax=axes[1, 0], palette="Blues_d")
axes[1, 0].set_title("Student-Test Log Loss (Lower is Better)", fontsize=12, weight="bold")

# 4. Latency
sns.barplot(data=df_results, x="model", y="total_train_time_s", ax=axes[1, 1], palette="rocket")
axes[1, 1].set_title("Total Training Latency (Seconds)", fontsize=12, weight="bold")

plt.tight_layout()
plt.savefig("results/model_comparison_overview.png", dpi=300)
plt.show()


## 8. Bayesian Hyperparameter Tuning with Optuna
LightGBM achieved the best overall performance, highest Student-Test Top-1 accuracy (**88.25%**), and lowest training latency (**~20 seconds**). We optimize its hyperparameters using Optuna (TPE Sampler) over 40 trials.


In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 350, step=50),
        "max_depth": trial.suggest_int("max_depth", 5, 12),
        "num_leaves": trial.suggest_int("num_leaves", 20, 127),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }
    
    gkf_opt = GroupKFold(n_splits=3)
    oof_p = np.zeros(len(X_train))
    for train_idx, val_idx in gkf_opt.split(X_train, y_train, groups=groups_train):
        clf = lgb.LGBMClassifier(**params)
        clf.fit(X_train.iloc[train_idx], y_train[train_idx])
        oof_p[val_idx] = clf.predict_proba(X_train.iloc[val_idx])[:, 1]
        
    metrics = evaluate_predictions(meta_train, y_train, oof_p)
    return metrics["macro_f1"]

# Load existing study if available or run
optuna_study_path = Path("tuning/optuna_study.pkl")
if optuna_study_path.exists():
    study = joblib.load(optuna_study_path)
    print(f"Loaded existing study with {len(study.trials)} trials.")
else:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=40)
    joblib.dump(study, "tuning/optuna_study.pkl")

print(f"Best Trial #{study.best_trial.number}: Macro F1 = {study.best_value:.4f}")
print("Best Parameters:", json.dumps(study.best_params, indent=2))


## 9. Final Model Training & Artifact Export
We retrain the best LightGBM model with the optimal hyperparameters on the entire 78-trip clean training dataset and export production artifacts for Members C and D.


In [ ]:
best_params = study.best_params.copy()
best_params.update({"class_weight": "balanced", "random_state": 42, "n_jobs": -1, "verbose": -1})

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X_train, y_train)

# Fit production preprocessing pipeline
pipeline = Pipeline([("scaler", StandardScaler())])
pipeline.fit(X_train)

# Verify on Student-Test
final_test_probs = final_model.predict_proba(X_test)[:, 1]
final_metrics = evaluate_predictions(meta_test, y_test, final_test_probs)
print(f"Final Model Student-Test Top-1 Accuracy: {final_metrics['top1_accuracy']:.4f}")
print(f"Final Model Student-Test Macro F1:       {final_metrics['macro_f1']:.4f}")

# Save artifacts
Path("artifacts").mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, "artifacts/model.joblib")
joblib.dump(pipeline, "artifacts/preprocessing_pipeline.joblib")

metadata = {
    "model_type": "LightGBM Classifier",
    "feature_columns": FEATURE_COLS,
    "classes": [0, 1],
    "training_rows": len(X_train),
    "training_trips": int(len(np.unique(groups_train))),
    "best_hyperparameters": study.best_params,
    "student_test_metrics": final_metrics
}
with open("artifacts/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Artifacts successfully saved to artifacts/model.joblib and artifacts/model_metadata.json!")


## 10. Integration Guide for Other Team Members

### **Member C (HMM / Viterbi Road Matching)**
Member C requires candidate emission probabilities $P(	ext{point}_i 	ext{ is on candidate arc } a)$.
```python
import joblib
model = joblib.load('artifacts/model.joblib')

# Extract candidate feature matrix in strict column order
X_cand = candidates[FEATURE_COLS]
emission_probs = model.predict_proba(X_cand)[:, 1]
```

### **Member D (FastAPI REST Service)**
Member D loads the model once at startup and scores incoming points in real-time:
```python
from fastapi import FastAPI
import joblib, pandas as pd

app = FastAPI(title="PathSnap GPS Matcher")
model = joblib.load("artifacts/model.joblib")

@app.post("/predict")
def predict_candidate(features: dict):
    df_in = pd.DataFrame([features])[FEATURE_COLS]
    prob = float(model.predict_proba(df_in)[0, 1])
    return {"match_probability": prob}
```
